# Late-highway board check for the (1e8, 1e9] decade

## Question and why it matters
The per-decade late-highway yield (Q19 in `notes/open-questions-hypotheses.md`) compares three decades. The two later ones
were rechecked on a 65536 board in October and lost 26 artefacts; the (1e8, 1e9] decade (12/854 = 1.41%) had not been.
This notebook re-runs its 12 highways on bigger boards and scores them by a criterion registered first.
Links: `README.md` (registration, results), `NOTEBOOK.md` (lab notebook), `experiments/archive/2026-10-late-highway-board-check/`.

## Assumptions and definitions
* A highway is *confirmed* on a board iff the run reports the census period, drift (dx, dy) and onset exactly (B2 criterion).
* *Board filler*: escape footprint >= 0.99 G in both directions (frozen from the earlier `classify.py`).
* The tiled backend's CSV equals the dense backend's (tests; checked here by the control `RRLRLR`). Status: held.

## Provenance
Prints the git commit, date and a SHA-256 for every input file. No seeds: blank-board deterministic runs.

In [1]:
import hashlib, subprocess, datetime
from pathlib import Path
import polars as pl

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "AGENTS.md").exists())
EXP = ROOT / "experiments/active/2026-11-late-highway-1e8-board-check"
INPUTS = [EXP / "results/census_rows.csv", EXP / "results/recheck_specs.txt", EXP / "results/classify.txt",
          EXP / "results/yields.txt", ROOT / "experiments/archive/2026-10-late-highway-board-check/results/control_65536.csv",
          *sorted((EXP / "results/recheck_65536").glob("*.csv")), *sorted((EXP / "results/recheck_32768").glob("*.csv")),
          *sorted((EXP / "results/explore").glob("*.csv"))]
assert all(p.exists() for p in INPUTS)
sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, capture_output=True, text=True).stdout.strip()
print("git", sha, "| run", datetime.datetime.now().isoformat(timespec="minutes"))
for p in INPUTS:
    print(hashlib.sha256(p.read_bytes()).hexdigest()[:16], p.relative_to(ROOT))

git cfdb53c | run 2026-11-09T09:17
d696c2f5626c249f experiments/active/2026-11-late-highway-1e8-board-check/results/census_rows.csv
19f1cd8b5c438f4b experiments/active/2026-11-late-highway-1e8-board-check/results/recheck_specs.txt
738eaf011faf6257 experiments/active/2026-11-late-highway-1e8-board-check/results/classify.txt
aa8a483379fa86cd experiments/active/2026-11-late-highway-1e8-board-check/results/yields.txt
0f8e85e754a45fc9 experiments/archive/2026-10-late-highway-board-check/results/control_65536.csv
e3bcc1bd54d4b6e3 experiments/active/2026-11-late-highway-1e8-board-check/results/recheck_65536/RLLRRLRLR.csv
13422cafb56911a3 experiments/active/2026-11-late-highway-1e8-board-check/results/recheck_65536/RLRLLLRLLL.csv
c78890b9b0a7f35d experiments/active/2026-11-late-highway-1e8-board-check/results/recheck_65536/RLRLRLRLLR.csv
9416eb7e292af260 experiments/active/2026-11-late-highway-1e8-board-check/results/recheck_65536/RLRRLLLRL.csv
0041580b44a76d9f experiments/active/2026-11-late-

## Data and method
Rules: the 12 rows with an onset in `experiments/archive/2026-10-census-sweeps/results/sweep_1B.csv` (16384 board),
plus the control `RRLRLR`. Each ran alone from a blank board with budget census onset + 1e8, on a tiled 65536 board
(primary) and a dense 32768 board (cross-check): `bash scripts/recheck.sh` (binary sha256 in the `recheck.log` files).

In [2]:
KEY = ["period", "dx", "dy", "onset"]
def load(d):
    return pl.concat([pl.read_csv(f, infer_schema_length=None) for f in sorted((EXP / d).glob("*.csv"))], how="vertical_relaxed") \
        .cast({k: pl.Int64 for k in KEY}, strict=False)
census = pl.read_csv(EXP / "results/census_rows.csv")
r65, r32 = load("results/recheck_65536"), load("results/recheck_32768")
t = census.select("rule", *KEY, "decade", "filler").join(
    r65.select("rule", *[pl.col(k).alias(k + "_65536") for k in KEY], pl.col("width").alias("w_65536")), on="rule", how="left")
t = t.with_columns(confirmed_65536=pl.all_horizontal([pl.col(k).eq_missing(pl.col(k + "_65536")) for k in KEY]))
t = t.join(r32.select("rule", *[pl.col(k).alias(k + "_32768") for k in KEY]), on="rule", how="left").with_columns(
    confirmed_32768=pl.all_horizontal([pl.col(k).eq_missing(pl.col(k + "_32768")) for k in KEY]))
with pl.Config(tbl_rows=20, tbl_width_chars=200, tbl_cols=20):
    print(t.select("rule", "decade", "period", "onset", "onset_65536", "w_65536", "filler", "confirmed_65536", "confirmed_32768"))
print("confirmed on 65536:", t["confirmed_65536"].sum(), "of", t.height)

shape: (12, 9)
┌────────────┬───────────┬────────┬───────────┬─────────────┬─────────┬────────┬─────────────────┬─────────────────┐
│ rule       ┆ decade    ┆ period ┆ onset     ┆ onset_65536 ┆ w_65536 ┆ filler ┆ confirmed_65536 ┆ confirmed_32768 │
│ ---        ┆ ---       ┆ ---    ┆ ---       ┆ ---         ┆ ---     ┆ ---    ┆ ---             ┆ ---             │
│ str        ┆ str       ┆ i64    ┆ i64       ┆ i64         ┆ i64     ┆ bool   ┆ bool            ┆ bool            │
╞════════════╪═══════════╪════════╪═══════════╪═════════════╪═════════╪════════╪═════════════════╪═════════════════╡
│ RLLRRLRLR  ┆ (1e8,1e9] ┆ 4      ┆ 822854955 ┆ null        ┆ 12831   ┆ false  ┆ false           ┆ false           │
│ RLRLLLRLLL ┆ (1e8,1e9] ┆ 362    ┆ 643494213 ┆ 643494213   ┆ 12038   ┆ false  ┆ true            ┆ true            │
│ RLRLRLRLLR ┆ (1e8,1e9] ┆ 392    ┆ 287240145 ┆ 287240145   ┆ 33733   ┆ false  ┆ true            ┆ true            │
│ RLRRLLLRL  ┆ (1e8,1e9] ┆ 520    ┆ 108307758 ┆ 1

## Results
The table shows 11 of 12 census highways reproduce exactly on both new boards; `RLLRRLRLR` has no highway within the budget
on either. It does not show that the 11 are highways on every board (only 16384, 32768, 65536 and, from `confirm`, 8192).

In [3]:
ctrl = r65.filter(pl.col("rule") == "RRLRLR").select(KEY).row(0)
ref = pl.read_csv(ROOT / "experiments/archive/2026-10-late-highway-board-check/results/control_65536.csv") \
    .filter(pl.col("rule") == "RRLRLR").select(KEY).row(0)
print("control RRLRLR tiled 65536:", ctrl, "| earlier dense 65536:", ref, "| equal:", ctrl == ref)
print((EXP / "results/yields.txt").read_text())

control RRLRLR tiled 65536: (220, 2, -2, 1076356698) | earlier dense 65536: (220, 2, -2, 1076356698) | equal: True
(1e8,1e9] found by 1e9 rerun: 11/854 = 1.29%
(1e8,1e9] by onset: 10/854 = 1.17%
(1e9,1e10]: 16/807 = 1.98%
(1e10,1e11]: 12/732 = 1.64%
Fisher exact (1e8,1e9] found by 1e9 rerun vs (1e9,1e10]: p = 0.33
Fisher exact (1e8,1e9] by onset vs (1e9,1e10]: p = 0.24
Fisher exact (1e9,1e10] vs (1e10,1e11]: p = 0.70
Fisher exact (1e8,1e9] found by 1e9 rerun vs (1e10,1e11]: p = 0.67



The control reproduces exactly, so the tiled run is a valid stand-in for dense here. The corrected yield 11/854 = 1.29%
(10/854 by onset binning) does not differ from the next decades (Fisher p = 0.33, 0.67); it does not exclude a slow decline.

In [4]:
ex = pl.concat([pl.read_csv(f, infer_schema_length=None).cast({k: pl.Int64 for k in KEY}, strict=False).with_columns(board=pl.lit(f.stem.split("_")[-1])) for f in sorted((EXP / "results/explore").glob("*.csv"))], how="vertical_relaxed")
print(ex.select("board", "stop", "steps", "period", "dx", "dy", "onset", "width", "height", "tail_period"))
print("RLLRRLRLR census onsets by board: 8192 -> 763,443,858 (confirm_1B), 16384 -> 822,854,955 (census)")

shape: (3, 10)
┌─────────┬─────────┬─────────────┬────────┬───┬────────────┬───────┬────────┬─────────────┐
│ board   ┆ stop    ┆ steps       ┆ period ┆ … ┆ onset      ┆ width ┆ height ┆ tail_period │
│ ---     ┆ ---     ┆ ---         ┆ ---    ┆   ┆ ---        ┆ ---   ┆ ---    ┆ ---         │
│ str     ┆ str     ┆ i64         ┆ i64    ┆   ┆ i64        ┆ i64   ┆ i64    ┆ i64         │
╞═════════╪═════════╪═════════════╪════════╪═══╪════════════╪═══════╪════════╪═════════════╡
│ 1048576 ┆ budget  ┆ 10000000000 ┆ null   ┆ … ┆ null       ┆ 84683 ┆ 83799  ┆ 4           │
│ 262144  ┆ budget  ┆ 10000000000 ┆ null   ┆ … ┆ null       ┆ 84683 ┆ 83799  ┆ 4           │
│ 65536   ┆ escaped ┆ 2118888449  ┆ 4      ┆ … ┆ 2118845739 ┆ 33446 ┆ 32563  ┆ 4           │
└─────────┴─────────┴─────────────┴────────┴───┴────────────┴───────┴────────┴─────────────┘
RLLRRLRLR census onsets by board: 8192 -> 763,443,858 (confirm_1B), 16384 -> 822,854,955 (census)


`RLLRRLRLR`'s period-4 'highway' has a different onset on every board (8192, 16384, 65536) and does not occur by 1e10 on
262144 or 1048576 (the last two were an unregistered exploratory follow-up). It looks like a ray emitter whose straight
runs end, read as highways only when one reaches the board edge. This is an interpretation, not tested.

## Caveats and controls
* Positive control `RRLRLR` reproduced; `cargo test --release --locked` passed (227 tests).
* P1 (11/11 confirmed) and P2 (RLLRRLRLR not reproduced) held as registered. The artefact had already been flagged by the
  2026-10 `confirm` run (`confirmed_1B.csv`); the 12/854 yield had ignored that flag.
* One extra board size for the score; dependence that appears only above 65536 would not be caught.

## Pruning decisions and reusable negative results
Index revision `6fba1c8`: no matching signatures (README). No new signature: a corrected count is not a dead end.

## What this changes, next steps, open questions
The (1e8, 1e9] yield becomes 11/854 = 1.29% (10/854 by onset). Q19's flat reading stands. Open: what `RLLRRLRLR`'s period-4
runs travel over and why they stop; whether it is the same class as `RLLLRLRRR`.